# Remote NDT — competency question test harness

Turns a filled-in data template into RDF, merges it with the RNDT ontology schema, and runs all
**23 competency questions** against it.

**To use it:** put your data in `rndt-data-template.xlsx` (start from `rndt-data-template-blank.xlsx`),
point the config cell below at it, then *Run All*. Every question reports pass/fail with the actual
answers it returned.

The template holds **instance data only** — your inspections, components, people, images. The taxonomy
(Process Stages, Major Actions, Action Areas, Activities, Authority Classes) lives in `rndt-ontology.ttl`
and is merged in automatically, so you never retype it and it cannot drift.

Input may be either the `.xlsx` **or** a directory of CSVs, one per sheet — both produce identical results.

*Requires only `rdflib` and `openpyxl`.*

In [1]:
# Setup — installs the two dependencies if they are missing, then loads helpers.
import sys, subprocess
for _pkg, _mod in [("rdflib", "rdflib"), ("openpyxl", "openpyxl")]:
    try:
        __import__(_mod)
    except ImportError:
        print(f"installing {_pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", _pkg])

import csv, os, re, html
from pathlib import Path
import rdflib
from rdflib import Literal, URIRef
from rdflib.namespace import RDF, RDFS, OWL, XSD
from IPython.display import HTML, display

RNDT = rdflib.Namespace("http://www.cranfield.ac.uk/rndt-ontology#")

_CSS = """
<style>
.rndt { font-family: ui-sans-serif, -apple-system, "Segoe UI", sans-serif; font-size: 13px;
        color: #1a1a18; margin: 4px 0 14px; }
.rndt .cap { font-size: 11px; text-transform: uppercase; letter-spacing: .06em;
             color: #8a8677; margin-bottom: 5px; font-weight: 600; }
.rndt table { border-collapse: collapse; background: #fff; border: 1px solid #e4e1d8;
              border-radius: 8px; overflow: hidden; }
.rndt th { text-align: left; font-size: 10px; text-transform: uppercase; letter-spacing: .05em;
           color: #8a8677; background: #f4f2ea; padding: 7px 11px; border-bottom: 1px solid #e4e1d8; }
.rndt td { padding: 6px 11px; border-bottom: 1px solid #efede4; color: #1a1a18;
           vertical-align: top; font-variant-numeric: tabular-nums; }
.rndt tr:last-child td { border-bottom: none; }
.rndt .mono { font-family: ui-monospace, "SF Mono", Consolas, monospace; font-size: 11.5px; }
.rndt .ok   { color: #1e6b30; font-weight: 600; }
.rndt .bad  { color: #a3341c; font-weight: 600; }
.rndt .warn { color: #8a6d3b; font-weight: 600; }
.rndt .dim  { color: #8a8677; }
.rndt pre { font-family: ui-monospace, "SF Mono", Consolas, monospace; font-size: 11px;
            background: #f8f7f2; border: 1px solid #ece9df; border-radius: 6px;
            padding: 8px 10px; margin: 5px 0; white-space: pre-wrap; color: #3a3934; }
.rndt .note { font-size: 12px; color: #57554c; margin: 4px 0; }
</style>
"""
_css_done = False

def table(headers, rows, caption=None, classes=None):
    """Render a small HTML table with the project styling."""
    global _css_done
    css = "" if _css_done else _CSS
    _css_done = True
    cls = classes or {}
    head = "".join(f"<th>{html.escape(str(h))}</th>" for h in headers)
    body = ""
    for r in rows:
        tds = ""
        for i, v in enumerate(r):
            c = cls.get(i, "")
            tds += f'<td class="{c}">{v if c.startswith("raw") else html.escape(str(v))}</td>'
        body += f"<tr>{tds}</tr>"
    cap = f'<div class="cap">{html.escape(caption)}</div>' if caption else ""
    return HTML(f'{css}<div class="rndt">{cap}<table><tr>{head}</tr>{body}</table></div>')

def note(msg, kind="note"):
    global _css_done
    css = "" if _css_done else _CSS
    _css_done = True
    return HTML(f'{css}<div class="rndt"><div class="{kind}">{msg}</div></div>')

print("ready —", "rdflib", rdflib.__version__)

ready — rdflib 7.6.0


In [2]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
# The only cell you normally need to edit.

DATA_IN   = "rndt-data-template.xlsx"      # .xlsx file, OR a directory of CSVs
SCHEMA    = "rndt-ontology.ttl"            # the taxonomy / TBox — merged in, never modified
TTL_OUT   = "rndt-generated.ttl"           # where the converted instance data is written

# ──────────────────────────────────────────────────────────────────────────────
for _p in (DATA_IN, SCHEMA):
    if not Path(_p).exists():
        raise FileNotFoundError(f"{_p!r} not found — check the path in this cell.")
print(f"input : {DATA_IN}")
print(f"schema: {SCHEMA}")
print(f"output: {TTL_OUT}")

input : rndt-data-template.xlsx
schema: rndt-ontology.ttl
output: rndt-generated.ttl


In [3]:
# Load the template — accepts an .xlsx workbook or a directory of CSVs (one per sheet).

DATA_SHEETS = ["Components", "Participants", "Facilities", "Software", "Equipment",
               "Procedures", "Specifications", "AcceptanceCriteria", "Inspections",
               "Images", "Reports", "Indications", "ActivityLog"]

def _clean(v):
    if v is None:
        return ""
    s = str(v).strip()
    return "" if s.lower() in ("none", "nan") else s

def load_xlsx(path):
    import openpyxl
    wb = openpyxl.load_workbook(path, data_only=True)
    out = {}
    for name in DATA_SHEETS:
        if name not in wb.sheetnames:
            out[name] = ([], [])
            continue
        ws = wb[name]
        rows = list(ws.iter_rows(values_only=True))
        if not rows:
            out[name] = ([], [])
            continue
        headers = [_clean(h) for h in rows[0] if _clean(h)]
        data = []
        for r in rows[1:]:
            vals = [_clean(v) for v in r[:len(headers)]]
            if any(vals):
                data.append(dict(zip(headers, vals)))
        out[name] = (headers, data)
    return out

def load_csvdir(path):
    out = {}
    for name in DATA_SHEETS:
        f = Path(path) / f"{name}.csv"
        if not f.exists():
            out[name] = ([], [])
            continue
        with open(f, newline="", encoding="utf-8-sig") as fh:
            rdr = list(csv.reader(fh))
        if not rdr:
            out[name] = ([], [])
            continue
        headers = [_clean(h) for h in rdr[0] if _clean(h)]
        data = []
        for r in rdr[1:]:
            vals = [_clean(v) for v in (list(r) + [""] * len(headers))[:len(headers)]]
            if any(vals):
                data.append(dict(zip(headers, vals)))
        out[name] = (headers, data)
    return out

DATA = load_csvdir(DATA_IN) if Path(DATA_IN).is_dir() else load_xlsx(DATA_IN)

_total = sum(len(v[1]) for v in DATA.values())
display(table(
    ["Sheet", "Rows", "Columns"],
    [[s, len(DATA[s][1]), ", ".join(DATA[s][0]) or "—"] for s in DATA_SHEETS],
    caption=f"Loaded {_total} data rows from {Path(DATA_IN).name}",
    classes={1: "mono"}))

Sheet,Rows,Columns
Components,8,"id, label, partNumber, serialNumber"
Participants,6,"id, name, role, authorityClass"
Facilities,2,"id, label"
Software,2,"id, label"
Equipment,2,"id, label"
Procedures,2,"id, label, revision, approvedBy"
Specifications,1,"id, label"
AcceptanceCriteria,2,"id, label"
Inspections,8,"id, label, date, disposition, component, procedure, acceptanceCriteria, equipment, specification, facility"
Images,24,"id, label, inspection, kind, derivedFrom, storageLocation, fileFormat, viewedWithSoftware"


In [4]:
# Validate before converting. Problems are reported, not raised — a partly filled template
# still converts, and you simply see which competency questions it cannot answer.

schema_g = rdflib.Graph(); schema_g.parse(SCHEMA, format="turtle")
_local = lambda u: str(u).split("#")[-1]
VOCAB_ACTIVITY  = {_local(s) for s in schema_g.subjects(RDF.type, RNDT.Activity)}
VOCAB_AUTHORITY = {_local(s) for s in schema_g.subjects(RDF.type, RNDT.AuthorityClass)}

# column -> sheet it must reference
REFS = {
    "Procedures":  {"approvedBy": "Participants"},
    "Inspections": {"component": "Components", "procedure": "Procedures",
                    "acceptanceCriteria": "AcceptanceCriteria", "equipment": "Equipment",
                    "specification": "Specifications", "facility": "Facilities"},
    "Images":      {"inspection": "Inspections", "derivedFrom": "Images",
                    "viewedWithSoftware": "Software"},
    "Reports":     {"inspection": "Inspections"},
    "Indications": {"image": "Images", "acceptanceCriteria": "AcceptanceCriteria"},
    "ActivityLog": {"inspection": "Inspections", "performedBy": "Participants"},
}
IDS = {s: {r.get("id", "") for r in DATA[s][1] if r.get("id")} for s in DATA_SHEETS}

issues = []
def flag(sheet, row, col, msg):
    issues.append([sheet, row, col, msg])

for sheet in DATA_SHEETS:
    headers, rows = DATA[sheet]
    if headers and "id" not in headers:
        flag(sheet, "—", "id", "sheet has no 'id' column")
    seen = {}
    for i, r in enumerate(rows, start=2):          # row 2 = first data row
        rid = r.get("id", "")
        if not rid:
            flag(sheet, i, "id", "missing id"); continue
        if not re.fullmatch(r"[A-Za-z0-9_\-]+", rid):
            flag(sheet, i, "id", f"invalid characters in id {rid!r} — use letters, digits, _ or -")
        if rid in seen:
            flag(sheet, i, "id", f"duplicate id {rid!r} (first seen on row {seen[rid]})")
        seen[rid] = i
        for col, target in REFS.get(sheet, {}).items():
            v = r.get(col, "")
            if v and v not in IDS[target]:
                flag(sheet, i, col, f"{v!r} is not an id in the {target} sheet")
        if sheet == "ActivityLog":
            a = r.get("activity", "")
            if a and a not in VOCAB_ACTIVITY:
                flag(sheet, i, "activity", f"{a!r} is not an Activity in {SCHEMA}")
        if sheet == "Participants":
            a = r.get("authorityClass", "")
            if a and a not in VOCAB_AUTHORITY:
                flag(sheet, i, "authorityClass", f"{a!r} is not an Authority Class in {SCHEMA}")
        for col in ("date",):
            v = r.get(col, "")
            if v and not re.fullmatch(r"\d{4}-\d{2}-\d{2}", v):
                flag(sheet, i, col, f"{v!r} is not a YYYY-MM-DD date")

if issues:
    display(table(["Sheet", "Row", "Column", "Problem"], issues,
                  caption=f"{len(issues)} issue(s) found — these rows will be skipped or left incomplete",
                  classes={1: "mono", 2: "mono"}))
else:
    display(note("✓ No problems found — every id is unique and every reference resolves.", "ok"))

In [5]:
# Convert to RDF and write the Turtle file.

g = rdflib.Graph()
g.bind("rndt", RNDT); g.bind("rdfs", RDFS); g.bind("owl", OWL); g.bind("xsd", XSD)

U   = lambda i: RNDT[i]
def add_lit(s, p, v, dt=None):
    if v: g.add((U(s), p, Literal(v, datatype=dt) if dt else Literal(v)))
def add_ref(s, p, v, valid):
    if v and v in valid: g.add((U(s), p, U(v)))

def typed(s, cls, label=None):
    g.add((U(s), RDF.type, OWL.NamedIndividual)); g.add((U(s), RDF.type, RNDT[cls]))
    if label: g.add((U(s), RDFS.label, Literal(label, lang="en")))

def rows(sheet): return DATA[sheet][1]

for r in rows("Components"):
    typed(r["id"], "PhysicalEntity", r.get("label"))
    add_lit(r["id"], RNDT.partNumber, r.get("partNumber"))
    add_lit(r["id"], RNDT.serialNumber, r.get("serialNumber"))

for r in rows("Participants"):
    typed(r["id"], "Participant", r.get("name"))
    add_lit(r["id"], RNDT.hasRole, r.get("role"))
    add_ref(r["id"], RNDT.memberOfAuthorityClass, r.get("authorityClass"), VOCAB_AUTHORITY)

for r in rows("Facilities"):          typed(r["id"], "Facility", r.get("label"))
for r in rows("Software"):            typed(r["id"], "Software", r.get("label"))
for r in rows("Equipment"):           typed(r["id"], "PhysicalEntity", r.get("label"))
for r in rows("Specifications"):      typed(r["id"], "InformationEntity", r.get("label"))
for r in rows("AcceptanceCriteria"):  typed(r["id"], "InformationEntity", r.get("label"))

for r in rows("Procedures"):
    typed(r["id"], "InformationEntity", r.get("label"))
    add_lit(r["id"], RNDT.procedureRevision, r.get("revision"))
    add_ref(r["id"], RNDT.approvedBy, r.get("approvedBy"), IDS["Participants"])

for r in rows("Inspections"):
    typed(r["id"], "RemoteNDTProcess", r.get("label"))
    add_lit(r["id"], RNDT.inspectionDate, r.get("date"), XSD.date)
    add_lit(r["id"], RNDT.disposition, r.get("disposition"))
    add_ref(r["id"], RNDT.usesEntity, r.get("component"), IDS["Components"])
    add_ref(r["id"], RNDT.usesProcedure, r.get("procedure"), IDS["Procedures"])
    add_ref(r["id"], RNDT.usesAcceptanceCriteria, r.get("acceptanceCriteria"), IDS["AcceptanceCriteria"])
    add_ref(r["id"], RNDT.usesEquipment, r.get("equipment"), IDS["Equipment"])
    add_ref(r["id"], RNDT.appliesSpecification, r.get("specification"), IDS["Specifications"])
    add_ref(r["id"], RNDT.performedAtFacility, r.get("facility"), IDS["Facilities"])

for r in rows("Images"):
    typed(r["id"], "InformationEntity", r.get("label"))
    add_ref(r["id"], RNDT.belongsToInspection, r.get("inspection"), IDS["Inspections"])
    add_ref(r["id"], RNDT.derivedFrom, r.get("derivedFrom"), IDS["Images"])
    add_lit(r["id"], RNDT.storageLocation, r.get("storageLocation"))
    add_lit(r["id"], RNDT.fileFormat, r.get("fileFormat"))
    add_ref(r["id"], RNDT.viewedWithSoftware, r.get("viewedWithSoftware"), IDS["Software"])
    # 'kind' is informational only and is deliberately not written to the ontology.

for r in rows("Reports"):
    typed(r["id"], "InformationEntity", r.get("label"))
    add_ref(r["id"], RNDT.belongsToInspection, r.get("inspection"), IDS["Inspections"])

for r in rows("Indications"):
    typed(r["id"], "InformationEntity", r.get("label"))
    add_ref(r["id"], RNDT.appliesAcceptanceCriteria, r.get("acceptanceCriteria"), IDS["AcceptanceCriteria"])
    img = r.get("image", "")
    if img and img in IDS["Images"]:
        g.add((U(img), RNDT.hasIndication, U(r["id"])))   # image → indication

for r in rows("ActivityLog"):
    g.add((U(r["id"]), RDF.type, OWL.NamedIndividual))
    add_ref(r["id"], RNDT.logsActivity, r.get("activity"), VOCAB_ACTIVITY)
    add_ref(r["id"], RNDT.logForInspection, r.get("inspection"), IDS["Inspections"])
    add_ref(r["id"], RNDT.logPerformedBy, r.get("performedBy"), IDS["Participants"])
    add_lit(r["id"], RNDT.logDate, r.get("date"), XSD.date)
    add_lit(r["id"], RNDT.logResult, r.get("result"))

g.serialize(destination=TTL_OUT, format="turtle")

_inds = len(set(g.subjects(RDF.type, OWL.NamedIndividual)))
display(table(["Metric", "Value"],
              [["Individuals written", _inds],
               ["Triples written", len(g)],
               ["Output file", TTL_OUT]],
              caption="Conversion complete", classes={1: "mono"}))

Metric,Value
Individuals written,189
Triples written,1117
Output file,rndt-generated.ttl


In [6]:
# The 23 competency questions. Each is standard SPARQL 1.1 and runs against your data
# merged with the ontology schema.

full = rdflib.Graph()
full.parse(SCHEMA, format="turtle")
for t in g:
    full.add(t)

P = """PREFIX rndt: <http://www.cranfield.ac.uk/rndt-ontology#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
"""

# ── WHICH ONE each question asks about ───────────────────────────────────────
# Every competency question is of the form "...for THIS image / THIS inspection".
# These are the subjects it asks about. Change an id here to re-ask the same
# question of a different image, inspection, component or inspector.
#   rndt:LOCALNAME  -> an individual from your data
#   "text"          -> a literal value
PARAMS = {
     1: ("image",      "rndt:PROC_001"),   2: ("image",      "rndt:PROC_001"),
     3: ("image",      "rndt:PROC_001"),   4: ("inspection", "rndt:INSP_001"),
     5: ("inspection", "rndt:INSP_001"),   6: ("inspection", "rndt:INSP_001"),
     7: ("inspection", "rndt:INSP_001"),   8: ("inspection", "rndt:INSP_001"),
     9: ("inspection", "rndt:INSP_001"),  10: ("image",      "rndt:PROC_001"),
    11: ("component",  "rndt:C_001"),     12: ("report",     "rndt:REPORT_001"),
    13: ("inspection", "rndt:INSP_001"),  14: ("inspection", "rndt:INSP_001"),
    15: ("image",      "rndt:PROC_001"),  16: ("image",      "rndt:PROC_001"),
    17: ("serial",     '"SN00123"'),      18: ("component",  "rndt:C_001"),
    19: ("inspector",  "rndt:P_INS_01"),  20: ("image",      "rndt:PROC_001"),
    21: ("indication", "rndt:IND_001"),   22: ("image",      "rndt:PROC_003"),
    23: ("image",      "rndt:PROC_001"),
}

# Each entry: (question, SPARQL template). {{VALUES}} is filled from PARAMS above.
CQ = {}
CQ[1] = ("Which component is associated with this radiographic image?", P + """
SELECT ?componentLabel WHERE {
  {{VALUES}}
  ?image rndt:belongsToInspection ?inspection .
  ?inspection rndt:usesEntity ?component .
  ?component rdfs:label ?componentLabel .
}""")
CQ[2] = ("Who interpreted this radiographic image?", P + """
SELECT ?interpreterLabel WHERE {
  {{VALUES}}
  ?image rndt:belongsToInspection ?inspection .
  ?log rndt:logsActivity rndt:IdentifyIndications ;
       rndt:logForInspection ?inspection ;
       rndt:logPerformedBy ?interpreter .
  ?interpreter rdfs:label ?interpreterLabel .
}""")
CQ[3] = ("What certification level did the interpreter hold?", P + """
SELECT ?authorityLabel WHERE {
  {{VALUES}}
  ?image rndt:belongsToInspection ?inspection .
  ?log rndt:logsActivity rndt:IdentifyIndications ;
       rndt:logForInspection ?inspection ;
       rndt:logPerformedBy ?interpreter .
  ?interpreter rndt:memberOfAuthorityClass ?authority .
  ?authority rdfs:label ?authorityLabel .
}""")
CQ[4] = ("Which procedure or work instruction was used for this inspection?", P + """
SELECT ?procedureLabel WHERE {
  {{VALUES}}
  ?inspection rndt:usesProcedure ?procedure .
  ?procedure rdfs:label ?procedureLabel .
}""")
CQ[5] = ("Which revision of the procedure was used?", P + """
SELECT ?revision WHERE {
  {{VALUES}}
  ?inspection rndt:usesProcedure ?procedure .
  ?procedure rndt:procedureRevision ?revision .
}""")
CQ[6] = ("Who approved the procedure used for the inspection?", P + """
SELECT ?approverLabel WHERE {
  {{VALUES}}
  ?inspection rndt:usesProcedure ?procedure .
  ?procedure rndt:approvedBy ?approver .
  ?approver rdfs:label ?approverLabel .
}""")
CQ[7] = ("What certification level did the procedure approver hold?", P + """
SELECT ?authorityLabel WHERE {
  {{VALUES}}
  ?inspection rndt:usesProcedure ?procedure .
  ?procedure rndt:approvedBy ?approver .
  ?approver rndt:memberOfAuthorityClass ?authority .
  ?authority rdfs:label ?authorityLabel .
}""")
CQ[8] = ("Which acceptance criteria were used for this inspection?", P + """
SELECT ?criteriaLabel WHERE {
  {{VALUES}}
  ?inspection rndt:usesAcceptanceCriteria ?criteria .
  ?criteria rdfs:label ?criteriaLabel .
}""")
CQ[9] = ("Which equipment was used for the inspection?", P + """
SELECT ?equipmentLabel WHERE {
  {{VALUES}}
  ?inspection rndt:usesEquipment ?equipment .
  ?equipment rdfs:label ?equipmentLabel .
}""")
CQ[10] = ("Which software was used to view or interpret the image?", P + """
SELECT ?softwareLabel WHERE {
  {{VALUES}}
  ?image rndt:viewedWithSoftware ?software .
  ?software rdfs:label ?softwareLabel .
}""")
CQ[11] = ("What is the inspection status of this component?", P + """
SELECT ?disposition WHERE {
  {{VALUES}}
  ?inspection rndt:usesEntity ?component ;
              rndt:disposition ?disposition .
}""")
CQ[12] = ("Which inspection produced a particular inspection report?", P + """
SELECT ?inspectionLabel WHERE {
  {{VALUES}}
  ?report rndt:belongsToInspection ?inspection .
  ?inspection rdfs:label ?inspectionLabel .
}""")
CQ[13] = ("Who produced the inspection report?", P + """
SELECT ?producerLabel WHERE {
  {{VALUES}}
  ?log rndt:logsActivity rndt:ApproveInspectionReport ;
       rndt:logForInspection ?inspection ;
       rndt:logPerformedBy ?producer .
  ?producer rdfs:label ?producerLabel .
}""")
CQ[14] = ("Which specification applies to this inspection?", P + """
SELECT ?specLabel WHERE {
  {{VALUES}}
  ?inspection rndt:appliesSpecification ?spec .
  ?spec rdfs:label ?specLabel .
}""")
CQ[15] = ("Where is a particular digital image stored?", P + """
SELECT ?storageLocation WHERE {
  {{VALUES}}
  ?image rndt:storageLocation ?storageLocation .
}""")
CQ[16] = ("What file format is used for a particular image?", P + """
SELECT ?fileFormat WHERE {
  {{VALUES}}
  ?image rndt:fileFormat ?fileFormat .
}""")
CQ[17] = ("Which images are associated with a particular serial number?", P + """
SELECT ?imageLabel WHERE {
  {{VALUES}}
  ?component rndt:serialNumber ?serial .
  ?inspection rndt:usesEntity ?component .
  ?image rndt:belongsToInspection ?inspection ;
         rndt:fileFormat ?format ;
         rdfs:label ?imageLabel .
}""")
CQ[18] = ("Which inspector interpreted images for a particular component?", P + """
SELECT ?inspectorLabel WHERE {
  {{VALUES}}
  ?inspection rndt:usesEntity ?component .
  ?log rndt:logsActivity rndt:IdentifyIndications ;
       rndt:logForInspection ?inspection ;
       rndt:logPerformedBy ?inspector .
  ?inspector rdfs:label ?inspectorLabel .
}""")
CQ[19] = ("Which components were interpreted by a particular inspector?", P + """
SELECT DISTINCT ?componentLabel WHERE {
  {{VALUES}}
  ?log rndt:logsActivity rndt:IdentifyIndications ;
       rndt:logForInspection ?inspection ;
       rndt:logPerformedBy ?inspector .
  ?inspection rndt:usesEntity ?component .
  ?component rdfs:label ?componentLabel .
}""")
CQ[20] = ("What inspection decision resulted from the interpretation?", P + """
SELECT ?disposition WHERE {
  {{VALUES}}
  ?image rndt:belongsToInspection ?inspection .
  ?inspection rndt:disposition ?disposition .
}""")
CQ[21] = ("Which acceptance criteria were applied to a particular indication?", P + """
SELECT ?criteriaLabel WHERE {
  {{VALUES}}
  ?indication rndt:appliesAcceptanceCriteria ?criteria .
  ?criteria rdfs:label ?criteriaLabel .
}""")
CQ[22] = ("What indications were identified in this image?", P + """
SELECT ?indicationLabel WHERE {
  {{VALUES}}
  ?image rndt:hasIndication ?indication .
  ?indication rdfs:label ?indicationLabel .
}""")
CQ[23] = ("Which facility interpreted the image?", P + """
SELECT ?facilityLabel WHERE {
  {{VALUES}}
  ?image rndt:belongsToInspection ?inspection .
  ?inspection rndt:performedAtFacility ?facility .
  ?facility rdfs:label ?facilityLabel .
}""")

def sparql_for(k):
    """Fill the {{VALUES}} placeholder from PARAMS."""
    var, term = PARAMS[k]
    return CQ[k][1].replace("{{VALUES}}", f"VALUES ?{var} {{ {term} }}")

def subject_of(k):
    """Human-readable description of what this question was asked about."""
    var, term = PARAMS[k]
    if term.startswith('"'):
        return var, term.strip('"'), ""
    local = term.split(":")[-1]
    lbl = full.value(RNDT[local], RDFS.label)
    if lbl is None:
        return var, local, "not in your data"
    return var, local, str(lbl)

RESULTS = {}
for k in sorted(CQ):
    try:
        RESULTS[k] = ("ok", [", ".join(str(x) for x in row) for row in full.query(sparql_for(k))])
    except Exception as e:
        RESULTS[k] = ("error", [str(e)])

_answered = sum(1 for k, (st, rows_) in RESULTS.items() if st == "ok" and rows_)
_rows = []
for k in sorted(CQ):
    st, res = RESULTS[k]
    if st == "error":
        badge, cls = "ERROR", "bad"
    elif res:
        badge, cls = "answered", "ok"
    else:
        badge, cls = "no data", "warn"
    preview = "; ".join(res[:2]) + ("  …" if len(res) > 2 else "")
    var, sid, slbl = subject_of(k)
    asked = (f'<span class="mono">?{var} = {html.escape(sid)}</span>'
             + (f'<br><span class="dim">{html.escape(slbl)}</span>' if slbl else ""))
    _rows.append([f"CQ-{k:02d}", CQ[k][0], asked, f'<span class="{cls}">{badge}</span>',
                  len(res) if st == "ok" else "—", preview])

display(table(["#", "Competency question", "Asked about", "Result", "Rows", "Answer"], _rows,
              caption=f"{_answered} of 23 questions answered by this data",
              classes={0: "mono", 2: "raw", 3: "raw", 4: "mono"}))

#,Competency question,Asked about,Result,Rows,Answer
CQ-01,Which component is associated with this radiographic image?,?image = PROC_001Processed radiographic image (INSP_001),answered,1,Turbine Blade Casting
CQ-02,Who interpreted this radiographic image?,?image = PROC_001Processed radiographic image (INSP_001),answered,1,David Whitfield
CQ-03,What certification level did the interpreter hold?,?image = PROC_001Processed radiographic image (INSP_001),answered,1,Level 2 Technical Evaluation Authority
CQ-04,Which procedure or work instruction was used for this inspection?,?inspection = INSP_001Digital Radiography inspection INSP_001,answered,1,Remote digital radiography procedure RNDT-DR-001
CQ-05,Which revision of the procedure was used?,?inspection = INSP_001Digital Radiography inspection INSP_001,answered,1,Rev C
CQ-06,Who approved the procedure used for the inspection?,?inspection = INSP_001Digital Radiography inspection INSP_001,answered,1,Thomas Okafor
CQ-07,What certification level did the procedure approver hold?,?inspection = INSP_001Digital Radiography inspection INSP_001,answered,1,Level 3 Governance and Assurance Authority
CQ-08,Which acceptance criteria were used for this inspection?,?inspection = INSP_001Digital Radiography inspection INSP_001,answered,1,Castings acceptance criteria ACC-CAST-B
CQ-09,Which equipment was used for the inspection?,?inspection = INSP_001Digital Radiography inspection INSP_001,answered,1,Digital radiography system DR-4500
CQ-10,Which software was used to view or interpret the image?,?image = PROC_001Processed radiographic image (INSP_001),answered,1,DICONDE image viewer v4.2


In [7]:
# Full detail — every question with its SPARQL and complete results.
# Useful as documentation of the queries, and for debugging a question that returned nothing.

_out = []
for k in sorted(CQ):
    q = CQ[k][0]
    sparql = sparql_for(k)
    var, sid, slbl = subject_of(k)
    st, res = RESULTS[k]
    if st == "error":
        body = f'<div class="bad">ERROR: {html.escape(res[0])}</div>'
    elif res:
        body = "<ul>" + "".join(f"<li>{html.escape(r)}</li>" for r in res) + "</ul>"
    else:
        body = ('<div class="warn">No rows — the data does not answer this question. '
                'Check the sheets listed for it in the CQ Coverage tab of the template.</div>')
    _out.append(
        f'<div style="margin:0 0 20px"><div class="cap">CQ-{k:02d}</div>'
        f'<div style="font-weight:600;margin-bottom:4px">{html.escape(q)}</div>'
        f'<div class="note">Asked about <span class="mono">?{var} = {html.escape(sid)}</span>'
        f'{" &middot; " + html.escape(slbl) if slbl else ""}</div>'
        f'<pre>{html.escape(sparql.strip())}</pre>{body}</div>')

display(HTML(_CSS + '<div class="rndt">' + "".join(_out) + "</div>"))